# EDA — Exploratory Data Analysis (探索性数据分析)

Interactive (Plotly) visualization of the **V3.1** feature dataset
(`data/convertData/V3.1_15min_features.csv`, 70 columns, 105,216 rows, 15-min 2023–2025).

Sections:
1. **Price patterns** — price over time, distribution, intraday & calendar patterns
2. **Weather & correlation** — temperature / wind vs price, correlation heatmap
3. **New V3.1 features** — nuclear power and cross-border grid flows

Every figure is rendered interactively **and** saved as a static PNG into `charts/`.


## 0. Setup (准备工作)


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# ── paths (this notebook lives in data_visualization/) ───────────────────
DATA_PATH   = Path('../data/convertData/V3.1_15min_features.csv')
CHARTS_DIR  = Path('../charts')
CHARTS_DIR.mkdir(exist_ok=True)

HELSINKI = 'Europe/Helsinki'


def show_and_save(fig, name, width=1200, height=600, scale=2):
    """Render a figure in the notebook and persist a static PNG into charts/."""
    fig.show()
    pio.write_image(fig, CHARTS_DIR / f'{name}.png', width=width, height=height, scale=scale)
    print(f'  saved -> {CHARTS_DIR / f"{name}.png"}')


In [ ]:
# Load the V3.1 feature table (15-min resolution, 2023-2025) and normalise the time axis
df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert(HELSINKI)
df = df.sort_values('datetime').reset_index(drop=True)

# calendar columns used by several plots
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month

print(f'Rows: {len(df):,} | {df["datetime"].min()} -> {df["datetime"].max()}')
print(f'Columns: {len(df.columns)}')


## 1. EDA — Price Patterns (价格规律)


In [ ]:
# 1.1 Price over time (sampled to keep the interactive chart responsive)
sample = df.sample(n=20000, random_state=42).sort_values('datetime')
fig = px.line(sample, x='datetime', y='price',
              title='1.1 Electricity Price Over Time (V3.1)',
              labels={'datetime': 'Time', 'price': 'Price (EUR/MWh)'})
show_and_save(fig, '1.1_price_over_time')

# 1.2 Price distribution
fig = px.histogram(df, x='price', nbins=120,
                   title='1.2 Price Distribution (V3.1)',
                   labels={'price': 'Price (EUR/MWh)', 'count': 'Count'})
fig.update_layout(showlegend=False)
show_and_save(fig, '1.2_price_distribution')


In [ ]:
# 1.3 Intraday pattern: price distribution by hour of day
fig = px.box(df, x='hour', y='price',
             title='1.3 Price by Hour of Day (V3.1)',
             labels={'hour': 'Hour', 'price': 'Price (EUR/MWh)'})
show_and_save(fig, '1.3_price_by_hour')


In [ ]:
# 1.4 Calendar pattern: average price by weekday and by month
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df['day_name'] = df['day_of_week'].map(dict(enumerate(day_names)))

avg_day = df.groupby('day_name', as_index=False)['price'].mean()
fig = px.bar(avg_day, x='day_name', y='price', category_orders={'day_name': day_names},
             title='1.4a Average Price by Weekday (V3.1)',
             labels={'day_name': 'Weekday', 'price': 'Avg Price (EUR/MWh)'})
show_and_save(fig, '1.4a_price_by_weekday')

avg_month = df.groupby('month', as_index=False)['price'].mean()
fig = px.bar(avg_month, x='month', y='price',
             title='1.4b Average Price by Month (V3.1)',
             labels={'month': 'Month', 'price': 'Avg Price (EUR/MWh)'})
show_and_save(fig, '1.4b_price_by_month')


## 2. EDA — Weather & Correlation (天气与相关性)


In [ ]:
# 1.5 Weather vs price
sample = df.sample(n=20000, random_state=42)

fig = px.scatter(sample, x='temp', y='price', opacity=0.3,
                 title='1.5a Temperature vs Price (V3.1)',
                 labels={'temp': 'Temperature (°C)', 'price': 'Price (EUR/MWh)'})
show_and_save(fig, '1.5a_temp_vs_price')

fig = px.scatter(sample, x='wind_speed', y='price', opacity=0.3,
                 title='1.5b Wind Speed vs Price (V3.1)',
                 labels={'wind_speed': 'Wind Speed (m/s)', 'price': 'Price (EUR/MWh)'})
show_and_save(fig, '1.5b_wind_vs_price')


In [ ]:
# 1.6 Correlation heatmap (price vs selected features, incl. new V3.1 features)
cols = ['price', 'temp', 'wind_speed', 'HDD', 'wind_power_proxy',
        'price_lag_1', 'price_lag_96', 'price_rolling_mean_1h', 'hour', 'day_of_week',
        'fi_total_net', 'fi_ee', 'nuclear_power_mw', 'nuclear_change_1d']
corr = df[cols].corr()

fig = px.imshow(corr, text_auto=True, aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='1.6 Correlation Heatmap (V3.1)')
show_and_save(fig, '1.6_correlation_heatmap')


## 3. EDA — New V3.1 Features (核电与跨境输电)


In [ ]:
# 1.7 Nuclear power vs price (new V3.1 feature)
sample = df.sample(n=20000, random_state=42).sort_values('datetime')

fig = px.line(sample, x='datetime', y='nuclear_power_mw',
              title='1.7a Nuclear Power Output Over Time (V3.1)',
              labels={'datetime': 'Time', 'nuclear_power_mw': 'Nuclear (MW)'})
show_and_save(fig, '1.7a_nuclear_over_time')

fig = px.scatter(sample, x='nuclear_power_mw', y='price', opacity=0.3,
                 title='1.7b Nuclear Output vs Price (V3.1)',
                 labels={'nuclear_power_mw': 'Nuclear (MW)', 'price': 'Price (EUR/MWh)'})
show_and_save(fig, '1.7b_nuclear_vs_price')


In [ ]:
# 1.8 Cross-border grid flows over time (new V3.1 features)
sample = df.sample(n=20000, random_state=42).sort_values('datetime')

fig = px.line(sample, x='datetime', y='fi_total_net',
              title='1.8a Finland Total Net Flow (MW) — positive=export',
              labels={'datetime': 'Time', 'fi_total_net': 'Net Flow (MW)'})
fig.add_hline(y=0, line_dash='dash', line_color='red')
show_and_save(fig, '1.8a_fi_total_net')

fig = px.line(sample, x='datetime', y='fi_ee',
              title='1.8b FI↔Estonia Estlink Flow (MW) — positive=export',
              labels={'datetime': 'Time', 'fi_ee': 'Estlink Flow (MW)'})
show_and_save(fig, '1.8b_fi_ee')
